# NB15 — Gradient Descent from Scratch

> **StatQuest: "Instead of solving the equation exactly, take small steps downhill until you reach the bottom of the bowl."**

---

## The main ideas:

1. Normal equations solve (XtX)b = Xty in ONE step — but for massive data, inverting XtX is too slow
2. Gradient Descent takes many small steps in the **downhill direction** of the SSR surface
3. Step size = **learning rate** (alpha) — too large -> overshoots; too small -> slow
4. Mini-batch SGD: compute gradient on a small batch -> faster, foundation of deep learning


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def flow_diagram(steps, title, colors=None, notes=None, figsize=(14, 2.8)):
    n = len(steps)
    default_colors = ['#1565C0','#2E7D32','#E65100','#6A1B9A',
                      '#00695C','#AD1457','#37474F','#4E342E',
                      '#0277BD','#558B2F','#C62828','#F57F17']
    colors = (colors or default_colors)[:n]
    notes  = notes or ['']*n
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(-0.3, n*3.1); ax.set_ylim(-1.2, 2.4); ax.axis('off')
    bw, bh = 2.6, 1.3
    for i,(step,color,note) in enumerate(zip(steps,colors,notes)):
        x = i*3.1
        box = FancyBboxPatch((x,0.2),bw,bh,boxstyle="round,pad=0.12",
                             facecolor=color,edgecolor='white',linewidth=1.5,alpha=0.90)
        ax.add_patch(box)
        ax.text(x+bw/2,0.2+bh/2,step,ha='center',va='center',fontsize=8.5,
                color='white',fontweight='bold',multialignment='center')
        if note:
            ax.text(x+bw/2,0.02,note,ha='center',va='top',fontsize=7,
                    color='#555',style='italic')
        if i < n-1:
            ax.annotate('',xy=(x+bw+0.38,0.2+bh/2),xytext=(x+bw+0.08,0.2+bh/2),
                       arrowprops=dict(arrowstyle='->',color='#444',lw=2.2))
    ax.set_title(title,fontsize=11,fontweight='bold',pad=6,color='#222')
    plt.tight_layout(pad=0.4); plt.show()

flow_diagram(
    steps=[
        'Initialise:\nb0=0, b1=0',
        'Compute SSR\nfor current\nb0, b1',
        'Compute\ngradients:\ndSSR/db0, dSSR/db1',
        'Update:\nb0 -= lr * grad_b0\nb1 -= lr * grad_b1',
        'Convergence?\nSSR not\ndecreasing?',
        'Return\nfinal b0, b1\n(OLS solution)',
    ],
    title='NB15 Conceptual Flow: Gradient Descent for OLS',
    colors=['#37474F','#1565C0','#2E7D32','#E65100','#6A1B9A','#C62828'],
)


## The gradient of SSR

```
SSR = sum(y_i - b0 - b1*x_i)^2

d(SSR)/d(b0) = -2 * sum(y_i - y_hat_i)           <- gradient w.r.t. intercept
d(SSR)/d(b1) = -2 * sum(x_i * (y_i - y_hat_i))   <- gradient w.r.t. slope

Update rule (gradient DESCENT = move OPPOSITE to gradient):
  b0 <- b0 - lr * d(SSR)/d(b0) / n
  b1 <- b1 - lr * d(SSR)/d(b1) / n
```

**Intuition:** if the gradient is positive (SSR increases as b0 increases), we decrease b0.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X = np.array([1.,2.,3.,4.,5.,6.,7.,8.,9.,10.])
y = np.array([40.,45.,50.,55.,65.,70.,75.,85.,90.,95.])
n = len(X)

def gradient_descent(X, y, lr=0.001, epochs=5000):
    b0, b1 = 0.0, 0.0
    history = {'ssr':[], 'b0':[], 'b1':[]}
    for _ in range(epochs):
        y_hat = b0 + b1*X
        resid = y - y_hat
        grad_b0 = -2*resid.mean()
        grad_b1 = -2*(X*resid).mean()
        b0 -= lr * grad_b0
        b1 -= lr * grad_b1
        history['ssr'].append(np.sum(resid**2))
        history['b0'].append(b0); history['b1'].append(b1)
    return b0, b1, history

b0_gd, b1_gd, hist = gradient_descent(X, y, lr=0.001, epochs=6000)

# OLS exact solution
xbar, ybar = X.mean(), y.mean()
b1_ols = np.sum((X-xbar)*(y-ybar))/np.sum((X-xbar)**2)
b0_ols = ybar - b1_ols*xbar

print(f"GD result:  b0={b0_gd:.5f}  b1={b1_gd:.5f}")
print(f"OLS exact:  b0={b0_ols:.5f}  b1={b1_ols:.5f}")
print(f"Close?      {np.allclose([b0_gd,b1_gd],[b0_ols,b1_ols],atol=0.01)}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].semilogy(hist['ssr'], color='steelblue', linewidth=1.5)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('SSR (log scale)')
axes[0].set_title('SSR decreasing each epoch — converging to minimum')
axes[0].grid(alpha=0.3)

axes[1].plot(hist['b0'][::50], hist['b1'][::50], 'o-', color='steelblue',
             markersize=4, linewidth=1, label='GD path (every 50th step)')
axes[1].plot(b0_ols, b1_ols, 'r*', markersize=16, label='OLS minimum')
axes[1].set_xlabel('b0'); axes[1].set_ylabel('b1')
axes[1].set_title('Path of GD through (b0, b1) space')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# Effect of learning rate
import numpy as np, matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, lr, label in zip(axes,
    [0.0001, 0.001, 0.003],
    ['Too slow (lr=0.0001)', 'Good (lr=0.001)', 'Diverges (lr=0.003)']):
    b0, b1 = 0., 0.
    ssrs = []
    for _ in range(3000):
        yhat = b0 + b1*X
        res  = y - yhat
        b0  -= lr * (-2*res.mean())
        b1  -= lr * (-2*(X*res).mean())
        ssrs.append(np.sum(res**2))
    ax.semilogy(ssrs, color='steelblue', linewidth=1.5)
    ax.set_title(label); ax.set_xlabel('Epoch'); ax.set_ylabel('SSR')
    ax.grid(alpha=0.3)

plt.suptitle('Learning rate: too small = slow, just right = converges, too large = diverges',
             fontsize=11)
plt.tight_layout(); plt.show()


## Normal Equations vs Gradient Descent

| | Normal Equations | Gradient Descent |
|--|-----------------|-----------------|
| Solution | Exact, one step | Approximate, many steps |
| Time | O(n * k^2) + O(k^3) | O(n * k * epochs) |
| When wins | Small/medium k | Very large n or k |
| Foundation of | Classical statistics | Deep learning |
| Requires tuning | No | Yes — learning rate |

**Next: NB16 — Robust regression when your data has outliers.**
